In [58]:
import boto3

from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_BREAK

In [37]:
JOB_ID = '7e0358a9-12c4-4bc0-b72b-c502d0dfa335'

In [38]:
dynamodb = boto3.resource('dynamodb')
jury_instructions_table = dynamodb.Table('JuryInstructions-dev')

In [39]:
def fetch_ddb_item(job_id: str) -> dict:
    response = jury_instructions_table.get_item(Key={
        'jury_instruction_id': job_id
    })

    return response.get('Item')

In [40]:
item = fetch_ddb_item(JOB_ID)
instructions = item.get('jury_instructions_text')

In [ ]:
doc = Document()
doc.add_heading('Jury Instructions', level=1)

for it in instructions:
    if not isinstance(it, dict):
        continue

    number = str(it.get('number', '')).strip()
    text = str(it.get('customized_text', '')).strip()

    if not (number or text):
        continue

    para = doc.add_paragraph()

    if number:
        para.add_run(f'{number}. ').bold = True

    para.add_run(text)

In [ ]:
doc.save('test.docx')

In [43]:
party_type = 'PLAINTIFF' # PLAINTIFF or DEFENDANT

In [60]:
document = Document()

for section in document.sections:
    section.top_margin = Inches(1)
    section.bottom_margin = Inches(1)
    section.left_margin = Inches(1)
    section.right_margin = Inches(1)

for i, instruction in enumerate(instructions):
    jury_instruction_item_number_paragraph = document.add_paragraph(f'{party_type}’S REQUESTED JURY INSTRUCTION NO. {i + 1}\n')

    run = jury_instruction_item_number_paragraph.runs[0]
    run.font.name = 'Times New Roman'
    run.font.size = Pt(12)
    run.font.bold = True

    jury_instruction_title_paragraph = document.add_paragraph(instruction['title'].upper())
    jury_instruction_title_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    run = jury_instruction_title_paragraph.runs[0]
    run.font.name = 'Times New Roman'
    run.font.size = Pt(12)
    run.font.bold = True

    jury_instruction_content_paragraph = document.add_paragraph(instruction['customized_text'])
    
    run = jury_instruction_content_paragraph.runs[0]
    run.font.name = 'Times New Roman'
    run.font.size = Pt(12)

    if instruction['number'].startswith('CUSTOM-DEFAMATION-'):
        instruction_number = f'Custom Jury Instruction {instruction["number"]}'
    else:
        instruction_number = f'Florida Standard Jury Instruction {instruction["number"]}'

    jury_instruction_number_paragraph = document.add_paragraph(instruction_number)

    run = jury_instruction_number_paragraph.runs[0]
    run.font.name = 'Times New Roman'
    run.font.size = Pt(12)
    run.font.italic = True
    
    jury_instruction_status_paragraph = document.add_paragraph('''
Granted ___________ 
Denied ___________ 
Withdrawn ___________'''.strip())
    
    run = jury_instruction_status_paragraph.runs[0]
    run.font.name = 'Times New Roman'
    run.font.size = Pt(12)

    run.add_break(WD_BREAK.PAGE)

document.save('test.docx')